# Load GitHub Issues Dataset from Hugging Face

This notebook loads the dataset from Hugging Face Hub instead of local CSV files.
Dataset: https://huggingface.co/datasets/sharjeelyunus/github-issues-dataset

In [11]:
import pandas as pd
import numpy as np
from datasets import load_dataset
import os

## Load Dataset from Hugging Face Hub

The dataset is downloaded and cached locally on first run.

In [12]:
# Load dataset from Hugging Face Hub
dataset = load_dataset('sharjeelyunus/github-issues-dataset')

print(f"Available splits: {list(dataset.keys())}")
print(f"\nDataset info:")
print(dataset)

Available splits: ['train']

Dataset info:
DatasetDict({
    train: Dataset({
        features: ['id', 'repo', 'title', 'body', 'labels', 'priority', 'severity'],
        num_rows: 114073
    })
})


## Explore Dataset Structure

In [13]:
# Get the first split (usually 'train')
split_name = list(dataset.keys())[0]
data_split = dataset[split_name]

print(f"Using split: {split_name}")
print(f"Number of examples: {len(data_split)}")
print(f"\nColumn names: {data_split.column_names}")
print(f"\nFirst example:")
print(data_split[0])

Using split: train
Number of examples: 114073

Column names: ['id', 'repo', 'title', 'body', 'labels', 'priority', 'severity']

First example:
{'id': 393061, 'repo': 'youtube-dl', 'title': 'Output file size with -s or -g', 'body': 'Was: http://bitbucket.org/rg3/youtube-dl/issue/141/\n\nIf the file size was outputted it would be possible to script youtube-dl to test if the current video the the harddrive is in the best possible quality.\n\nIt happens that Youtube re-encode videos, and with higher resolution, so being able to extract the file size from youtube-dl would be very useful.\n\nFile size in bytes would be preferred.\n', 'labels': 'request', 'priority': 'medium', 'severity': 'Critical'}


## Convert to Pandas DataFrame

Convert Hugging Face dataset to pandas for easier manipulation.

In [14]:
# Convert to pandas DataFrame
df = data_split.to_pandas()

print(f"DataFrame shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nFirst few rows:")
df.head()

DataFrame shape: (114073, 7)

Columns: ['id', 'repo', 'title', 'body', 'labels', 'priority', 'severity']

Data types:
id           int64
repo        object
title       object
body        object
labels      object
priority    object
severity    object
dtype: object

First few rows:


,id,repo,title,body,labels,priority,severity
0,393061,youtube-dl,Output file size with -s or -g,Was: http://bitbucket.org/rg3/youtube-dl/issue...,request,medium,Critical
1,1637737,youtube-dl,Create a php API and demo page,youtube-dl is often embedded by php applicatio...,php,low,Major
2,1639054,youtube-dl,"integrate template ""special sequences"" in help...",like in http://rg3.github.com/youtube-dl/docum...,request,low,Minor
3,1789251,youtube-dl,Add a path option to --keep-video,"Hey there,\n\nI think it would be a great idea...",request,low,Minor
4,1789512,youtube-dl,add support for picasaweb.google.com video clips,> /opt/local/bin/youtube-dl -t https://picasaw...,site-support-request,low,Minor


## Explore Dataset Content

In [15]:
print("Dataset Statistics:")
print(f"Shape: {df.shape}")
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Dataset Statistics:
Shape: (114073, 7)

Missing values:
id            0
repo          0
title         0
body        133
labels        0
priority      0
severity      0
dtype: int64

Duplicate rows: 0

Memory usage: 621.80 MB


## Check Available Columns

The dataset might have different column names than expected.

In [16]:
# Display all columns
print("Available columns:")
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col}")

print(f"\nSample data from each column:")
for col in df.columns:
    print(f"\n{col}:")
    print(f"  Type: {df[col].dtype}")
    print(f"  Non-null: {df[col].notna().sum()}")
    if df[col].dtype == 'object':
        print(f"  Sample: {df[col].iloc[0][:100] if len(str(df[col].iloc[0])) > 100 else df[col].iloc[0]}")

Available columns:
  1. id
  2. repo
  3. title
  4. body
  5. labels
  6. priority
  7. severity

Sample data from each column:

id:
  Type: int64
  Non-null: 114073

repo:
  Type: object
  Non-null: 114073
  Sample: youtube-dl

title:
  Type: object
  Non-null: 114073
  Sample: Output file size with -s or -g

body:
  Type: object
  Non-null: 113940
  Sample: Was: http://bitbucket.org/rg3/youtube-dl/issue/141/

If the file size was outputted it would be poss

labels:
  Type: object
  Non-null: 114073
  Sample: request

priority:
  Type: object
  Non-null: 114073
  Sample: medium

severity:
  Type: object
  Non-null: 114073
  Sample: Critical


## Prepare Data for Model Pipeline

Extract relevant columns and ensure compatibility with existing pipeline.

In [17]:
# Identify the relevant columns for text and severity
# Common naming patterns in the dataset
text_columns = [col for col in df.columns if col.lower() in ['text', 'body', 'title', 'issue', 'description']]
severity_columns = [col for col in df.columns if col.lower() in ['severity', 'label', 'priority', 'importance']]

print(f"Potential text columns: {text_columns}")
print(f"Potential severity columns: {severity_columns}")

# If specific columns found, use them
if text_columns:
    text_col = text_columns[0]
    print(f"\nUsing text column: {text_col}")
else:
    print("\nAvailable columns:")
    print(df.columns.tolist())
    print("\nPlease specify which column contains the issue text.")

Potential text columns: ['title', 'body']
Potential severity columns: ['priority', 'severity']

Using text column: title


## Inspect Data Distribution

In [18]:
# Show value counts for categorical columns
print("Value counts for each column:")
for col in df.select_dtypes(include=['object']).columns:
    print(f"\n{col}:")
    print(df[col].value_counts())

# Show basic statistics
print(f"\nBasic statistics:")
df.describe()

Value counts for each column:

repo:
repo
pytorch                  14451
flutter                  13130
godot                    11207
rust                      9925
go                        9094
                         ...  
nodebestpractices            3
clean-code-javascript        2
fucking-algorithm            2
awesome                      1
awesome-go                   1
Name: count, Length: 68, dtype: int64

title:
title
[Bug]:                                                                                         23
Illegal value for lineNumber                                                                   14
Model is disposed!                                                                              6
Soft Assertion Failed                                                                           6
[DevTools Bug] Cannot add node "1" because a node with that id is already in the Store.         5
                                                                           

,id
count,1.140730e+05
mean,1.472725e+09
std,8.016439e+08
min,3.930610e+05
25%,7.523763e+08
50%,1.467228e+09
75%,2.177024e+09
max,2.816851e+09


## Save Processed Data

Save the loaded data for use in downstream notebooks.

In [19]:
# Create output directory if it doesn't exist
output_dir = '../data/raw/'
os.makedirs(output_dir, exist_ok=True)

# Save the original data from Hugging Face
csv_path = os.path.join(output_dir, 'github_issues_hf.csv')
df.to_csv(csv_path, index=False)
print(f"Saved to: {csv_path}")
print(f"File size: {os.path.getsize(csv_path) / 1024**2:.2f} MB")

# Also save as parquet for faster loading
parquet_path = os.path.join(output_dir, 'github_issues_hf.parquet')
df.to_parquet(parquet_path, index=False)
print(f"\nSaved to: {parquet_path}")
print(f"File size: {os.path.getsize(parquet_path) / 1024**2:.2f} MB")

Saved to: ../data/raw/github_issues_hf.csv
File size: 293.37 MB

Saved to: ../data/raw/github_issues_hf.parquet
File size: 131.29 MB


## Dataset Information

The dataset has been loaded from Hugging Face and saved locally.

In [20]:
print("Dataset Loading Summary:")
print(f"Source: https://huggingface.co/datasets/sharjeelyunus/github-issues-dataset")
print(f"\nLoaded data:")
print(f"  - Rows: {len(df)}")
print(f"  - Columns: {len(df.columns)}")
print(f"  - Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nSaved locally to:")
print(f"  - CSV: {csv_path}")
print(f"  - Parquet: {parquet_path}")
print(f"\nNext step: Use 02_preprocessing.ipynb to process this data")

Dataset Loading Summary:
Source: https://huggingface.co/datasets/sharjeelyunus/github-issues-dataset

Loaded data:
  - Rows: 114073
  - Columns: 7
  - Memory: 759.31 MB

Saved locally to:
  - CSV: ../data/raw/github_issues_hf.csv
  - Parquet: ../data/raw/github_issues_hf.parquet

Next step: Use 02_preprocessing.ipynb to process this data
